In [3]:
import pandas as pd

Load original pick data CSV into dataframe.

In [4]:
filename = r"C:\Users\LindseyBuss\Documents\OBETA_Project\003 pick_data.csv"
column_names = ['product_id', 'warehouse_section', 'origin', 'order_number', 'position_in_order', 'pick_volume', 'quantity_unit', 'date']
data_types = {
    'product_id': 'string', 
   'warehouse_section': 'category',
   'origin': 'category',
   'order_number': 'string',
   'position_in_order':'int64',
   'pick_volume': 'int64',
   'quantity_unit': 'string',
   'date': 'string'
}
df = pd.read_csv(filename, names=column_names, header=None, dtype=data_types, parse_dates=['date'])
print(df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  
0           29            St 2017-06-30 11:15:24  
1           30            St 2017-06-30 11:22:35  
2           30            St 2017-06-30 12:04:50  
3           20            St 2017-06-30 12:04:51  
4           30            St 2017-06-30 12:05:02  


In [5]:
# Add unique pick IDs.
df['pick_id'] = range(1, len(df) + 1)

# Extract year of order to create new unique order IDs.
df['year_of_order'] = df['date'].dt.year.astype(str)

df['updated_order_number'] = df[['order_number', 'year_of_order']].astype(str).agg('-'.join, axis=1)
print(df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  pick_id year_of_order  \
0           29            St 2017-06-30 11:15:24        1          2017   
1           30            St 2017-06-30 11:22:35        2          2017   
2           30            St 2017-06-30 12:04:50        3          2017   
3           20            St 2017-06-30 12:04:51        4          2017   
4           30            St 2017-06-30 12:05:02        5          2017   

  updated_order_number  
0        07055448-2017  
1        07055448-2017  
2        07055448-2017  
3        0

In [7]:
# Drop null values, duplicates and picks with a volume of zero.
df = df.dropna()
df = df.drop_duplicates()
df = df[df['pick_volume'] != 0]

print(df.describe())

       position_in_order   pick_volume                           date  \
count       3.369872e+07  3.369872e+07                       33698719   
mean        5.427194e+00  6.193287e+01  2016-06-30 07:23:20.040092928   
min         1.000000e+00 -2.000000e+03            2011-06-23 00:00:01   
25%         1.000000e+00  1.000000e+00     2014-05-06 18:01:56.500000   
50%         3.000000e+00  5.000000e+00            2016-10-21 18:49:00   
75%         7.000000e+00  2.400000e+01            2018-09-27 18:24:00   
max         4.360000e+02  2.000000e+05            2020-07-14 11:42:01   
std         6.953640e+00  3.667303e+02                            NaN   

            pick_id  
count  3.369872e+07  
mean   1.692005e+07  
min    1.000000e+00  
25%    8.429708e+06  
50%    1.686564e+07  
75%    2.543645e+07  
max    3.388899e+07  
std    9.798350e+06  


Subset data and prepare order summary information.

In [9]:
years = df['year_of_order'].unique()
years

array(['2017', '2018', '2020', '2019', '2013', '2012', '2011', '2014',
       '2015', '2016'], dtype=object)

In [ ]:
def summarize_year(df):
    

In [8]:
df_2019 = df[df['year_of_order'] == '2019']

order_summary_data = df_2019.groupby('updated_order_number').agg(
    origin = ('origin', 'first'),
    num_picks = ('pick_volume', 'sum'),
    num_products = ('product_id', 'nunique'),
    num_sections = ('warehouse_section', 'nunique'),
    num_positions = ('position_in_order', 'nunique'),
    time_of_first_pick = ('date', 'min'),
    time_of_last_pick = ('date', 'max'))

order_summary_data['time_to_fulfil'] = order_summary_data['time_of_last_pick'] - order_summary_data['time_of_first_pick']
order_summary_data['date'] = order_summary_data['time_of_first_pick'].dt.date

print(order_summary_data.head())

                     origin  num_picks  num_products  num_sections  \
updated_order_number                                                 
01000002-2019            48         13             5             1   
01000003-2019            48         87             6             2   
01000005-2019            48          1             1             1   
01000007-2019            46         44             4             3   
01000008-2019            48       2079            10             3   

                      num_positions  time_of_first_pick   time_of_last_pick  \
updated_order_number                                                          
01000002-2019                     5 2019-05-22 22:07:30 2019-05-22 22:07:31   
01000003-2019                     6 2019-05-22 23:56:31 2019-05-23 01:36:50   
01000005-2019                     1 2019-05-22 13:58:55 2019-05-22 13:58:55   
01000007-2019                     4 2019-05-22 14:01:57 2019-05-22 21:37:01   
01000008-2019                    10